## OSRM Network Query

Key files needed: 

| File | Used for |
|---|---|
| output/hdb_blocks.gpkg | Residential points - my origins | 
| data/ChildCareServices_2021.geojson| Residential points - my destinations| 
| data/MasterPlanSubZones_2019.geojson | Study area |

Key Dependencies:

| Library | Purpose |
|---|---|
| `xlogit` | Model estimation |
| `pandas` | Tabular data manipulation |
| `matplotlib` | QA scatter plots |
| `pickle` | Save and export models |
| `numpy` | Data processing |
| `statsmodels` | |
| `seaborn` | Graphs for checking correlation between model variables|
| `folium` | Visualisation of routes|


The extracted OSRM Singapore files can also be directly downloaded here. 
Data for running OSRM via Docker:
https://www.dropbox.com/scl/fo/de4tvoq346ylpexiz2k2s/AAKRUqoSCLjPv95NMsHSMMM?rlkey=sddbtns26x7skqf0d8nh98yak&st=3ytafdb2&dl=0

But, if you prefer to extract the data yourself, please follow the manual extraction instructions:


--- 

### Manual Extraction

1. First, please obtain the OSRM files through running osrm-extract in Docker on the singapore_only osm.pbf file in your Shell/Terminal:

OSRM uses Lua profiles (such as car.lua, bicycle.lua, or foot.lua) to determine which roads are passable for a specific transport mode. Use the osrm-extract tool to parse the PBF data.

```bash
docker run -t -v "${PWD}:/data" osrm/osrm-backend osrm-extract -p /opt/car.lua /data/singapore.osm.pbf
```

(Replace with *walk* for the walking profiles)

2. Next, prepare the data for the Multi-Level Dijkstra (MLD) routing algorithm. This requires partitioning the map into cells and calculating routing weights.

#### Partition the graph into cells
```bash
docker run -t -v "${PWD}:/data" osrm/osrm-backend osrm-partition /data/singapore-latest.osrm`
```

#### Customize the cells with routing weights
```bash
docker run -t -v "${PWD}:/data" osrm/osrm-backend osrm-customize /data/singapore-latest.osrm`
```

### Local OSRM routing servers

Accessibility costs (\(C_{i,j}^{m}\)) are computed via local OSRM instances, one per mode. Each assumes the corresponding `.osrm` graph has already been built (`osrm-extract` → `osrm-partition` → `osrm-customize`) into the mounted data folder.

| Mode    | Port | Data folder       |
|---------|------|--------------------|
| Walking | 5001 | `OSRMData_Walk`    |
| Driving | 5002 | `OSRMData_Drive`   |



```bash
# Walking — port 5001
docker run --volume="<your_path>/OSRMData_Walk:/data" -p 5001:5000 -d osrm/osrm-backend
```

```bash
# Driving — port 5002
docker run --volume="<your_path>/OSRMData_Drive:/data" -p 5002:5000 -d osrm/osrm-backend
\`\`\`

# ─────────────────────────────────────────────

### Notes for BT4015 Analysis
How to calculate timings between each origin + destination?
###
Neighbourhood - its closest childcare centre 

In [2]:
#origin - destination matrix

import requests
import numpy as np
import pandas as pd
import geopandas as gpd
import seaborn as sns

# ---- funciones auxiliares ----
def obtener_matriz_osrm(origenes, destinos, modo, tipo):
    """
    Obtiene una matriz OSRM entre 'origenes' y 'destinos':
    tipo='duration' devuelve minutos; tipo='distance' devuelve metros.
    """
    if tipo not in ('duration', 'distance'):
        raise ValueError("tipo must be 'duration' or 'distance'")

    # combinar orígenes y destinos en una sola lista
    coords = origenes + destinos
    coords_str = ';'.join([f"{lon},{lat}" for lon, lat in coords])

    # crear índices para OSRM (origen y destino)
    indices_origenes = ';'.join([str(i) for i in range(len(origenes))])
    indices_destinos = ';'.join([str(i) for i in range(len(origenes), len(coords))])

    # definir la url según el modo
    if modo == 'drive':
        url = f"http://localhost:5002/table/v1/driving/{coords_str}"
    else:  # modo == 'walk'
        url = f"http://localhost:5001/table/v1/foot/{coords_str}"

    params = {
        'sources': indices_origenes,
        'destinations': indices_destinos,
        'annotations': tipo  # 'duration' o 'distance'
    }

    response = requests.get(url, params=params)
    if response.status_code == 200:
        data = response.json()
        key = 'durations' if tipo == 'duration' else 'distances'
        # None = par inalcanzable -> np.inf
        values_clean = [
            [val if val is not None else np.inf for val in row]
            for row in data[key]
        ]
        values = np.array(values_clean, dtype=float)
        # Convertir segundos a minutos; mantener distancias en metros.
        return values / 60 if tipo == 'duration' else values
    else:
        # NOTE: a failed request fills this block with np.inf, which becomes impedance 0
        # downstream. Make sure both OSRM servers are running and watch for these
        # messages before exporting anything.
        print(f"Error: {response.status_code}")
        print(response.text)
        return np.full((len(origenes), len(destinos)), np.inf)

def dividir_lista(coordenadas, max_tam):
    """
    generador que divide la lista 'coordenadas' en sub-listas
    de longitud <= max_tam.
    """
    for i in range(0, len(coordenadas), max_tam):
        yield coordenadas[i:i + max_tam]


### calculating timings for walk matrix

In [3]:
#files

hdbs = gpd.read_file("../large_data/hdb_points_TEST.geojson")
ccs = gpd.read_file("../data/ChildCareServices_2021.geojson")

print(f"Loaded {len(hdbs)} HDB points and {len(ccs)} childcare services.")

Loaded 10741 HDB points and 1925 childcare services.


In [4]:
from tqdm.auto import tqdm

# copiar celdas para orígenes y destinos
gdf_origenes = hdbs.copy()
gdf_destinos = ccs.copy()

# convertir a EPSG:32719 y extraer centroides para cada uno, luego pasar a EPSG:4326
gdf_origenes = gdf_origenes.to_crs(32719)
gdf_origenes["geometry"] = gdf_origenes.geometry.centroid
gdf_origenes = gdf_origenes.to_crs(4326)

gdf_destinos = gdf_destinos.to_crs(32719)
gdf_destinos["geometry"] = gdf_destinos.geometry.centroid
gdf_destinos = gdf_destinos.to_crs(4326)

# extraer coordenadas de orígenes y destinos
coords_origenes = [(geom.x, geom.y) for geom in gdf_origenes.geometry]
coords_destinos = [(geom.x, geom.y) for geom in gdf_destinos.geometry]

# definir el tamaño máximo de 'chunk'
max_tam = 100

# crear una matriz completa llena de infinitos, de tamaño (num_origenes x num_destinos)
matriz_resultado_time = np.full((len(coords_origenes), len(coords_destinos)), np.inf)
matriz_resultado_dist = np.full((len(coords_origenes), len(coords_destinos)), np.inf)

# dividir orígenes y destinos en sub-listas
origen_chunks = list(dividir_lista(coords_origenes, max_tam))
destino_chunks = list(dividir_lista(coords_destinos, max_tam))

num_origenes = len(coords_origenes)
num_destinos = len(coords_destinos)
total_queries = 2 * len(origen_chunks) * len(destino_chunks)
print(f"Origins: {num_origenes:,} | Destinations: {num_destinos:,}")
print(f"Origin-destination pairs: {num_origenes * num_destinos:,} | OSRM queries: {total_queries:,}")

# Each batch makes two requests: duration and distance.
with tqdm(total=total_queries, desc="OSRM queries", unit="query") as progress:
    for i, sub_origenes in enumerate(origen_chunks):
        for j, sub_destinos in enumerate(destino_chunks):
            ini_i = i * max_tam
            fin_i = ini_i + len(sub_origenes)
            ini_j = j * max_tam
            fin_j = ini_j + len(sub_destinos)
            progress.set_postfix_str(
                f"origins {ini_i + 1:,}-{fin_i:,}/{num_origenes:,}, "
                f"destinations {ini_j + 1:,}-{fin_j:,}/{num_destinos:,}"
            )

            sub_matriz_t = obtener_matriz_osrm(sub_origenes, sub_destinos, modo='foot', tipo='duration')
            matriz_resultado_time[ini_i:fin_i, ini_j:fin_j] = sub_matriz_t
            progress.update(1)

            sub_matriz_d = obtener_matriz_osrm(sub_origenes, sub_destinos, modo='foot', tipo='distance')
            matriz_resultado_dist[ini_i:fin_i, ini_j:fin_j] = sub_matriz_d
            progress.update(1)

# convertir la matriz (numpy array) en un DataFrame de pandas, con index de orígenes y columns de destinos
walk_matrix_time = pd.DataFrame(matriz_resultado_time, index=gdf_origenes.index, columns=gdf_destinos.index)
walk_matrix_dist = pd.DataFrame(matriz_resultado_dist, index=gdf_origenes.index, columns=gdf_destinos.index)

# export matrix to json for use 
walk_matrix_time.to_json("walk_matrix_time.json", orient="split")
walk_matrix_dist.to_json("walk_matrix_dist.json", orient="split")

Origins: 10,741 | Destinations: 1,925
Origin-destination pairs: 20,676,425 | OSRM queries: 4,320


OSRM queries: 100%|██████████| 4320/4320 [03:47<00:00, 18.98query/s, origins 10,701-10,741/10,741, destinations 1,901-1,925/1,925]


# ─────────────────────────────────────────────

## Route Visualisation - Folium
Visualising shortest paths from each hdb centroid to each childhood centre

In [8]:
# Map a reproducible sample of HDBs and their nearest childcare by walking time.
import folium
from folium.plugins import MarkerCluster
from html import escape
from tqdm.auto import tqdm

MAX_ORIGINS = 100  # Set to None to show every origin (many route requests).
OSRM_WALK_URL = "http://localhost:5001"

if not coords_origenes or not coords_destinos:
    raise ValueError("Run the matrix cell first with non-empty origins and destinations.")
if (walk_matrix_time.shape != (len(coords_origenes), len(coords_destinos))
        or not walk_matrix_time.index.equals(gdf_origenes.index)
        or not walk_matrix_time.columns.equals(gdf_destinos.index)):
    raise ValueError("The walking matrix does not match the current points. Rerun the matrix cell.")

origin_positions = np.arange(len(coords_origenes))
if MAX_ORIGINS is not None and len(origin_positions) > MAX_ORIGINS:
    origin_positions = np.sort(np.random.default_rng(42).choice(
        origin_positions, size=MAX_ORIGINS, replace=False
    ))

# OSRM coordinates are (longitude, latitude); Folium uses (latitude, longitude).
map_center = [np.mean([lat for lon, lat in coords_origenes]),
              np.mean([lon for lon, lat in coords_origenes])]
m = folium.Map(location=map_center, zoom_start=12, tiles="OpenStreetMap")
origin_layer = MarkerCluster(name="HDB origins (displayed sample)").add_to(m)
destination_layer = MarkerCluster(name="Childcare centres (all)").add_to(m)
route_layer = folium.FeatureGroup(name="Routes to nearest childcare by time").add_to(m)

for j, (lon, lat) in enumerate(coords_destinos):
    label = escape(str(gdf_destinos.index[j]))
    folium.Marker(
        [lat, lon], popup=f"Childcare index: {label}",
        icon=folium.Icon(color="green", icon="info-sign")
    ).add_to(destination_layer)

print(f"Displaying {len(origin_positions):,}/{len(coords_origenes):,} HDB origins "
      f"and {len(coords_destinos):,} childcare destinations.")
routes_drawn = 0
no_finite_time = 0
route_errors = []
with requests.Session() as session:
    for i in tqdm(origin_positions, desc="Loading walking routes", unit="origin"):
        origin = coords_origenes[i]
        origin_label = escape(str(gdf_origenes.index[i]))
        times = walk_matrix_time.iloc[i].to_numpy(dtype=float)
        finite_positions = np.flatnonzero(np.isfinite(times))
        popup = f"HDB index: {origin_label}"
        if len(finite_positions):
            j = int(finite_positions[np.argmin(times[finite_positions])])
            destination_label = escape(str(gdf_destinos.index[j]))
            popup += (f"<br>Nearest childcare index: {destination_label}"
                      f"<br>Matrix walking time: {times[j]:.1f} min")
            destination = coords_destinos[j]
            coordinates = f"{origin[0]},{origin[1]};{destination[0]},{destination[1]}"
            try:
                response = session.get(
                    f"{OSRM_WALK_URL}/route/v1/foot/{coordinates}",
                    params={"overview": "full", "geometries": "geojson"},
                    timeout=30,
                )
                response.raise_for_status()
                data = response.json()
                if data.get("code") != "Ok" or not data.get("routes"):
                    raise ValueError(data.get("message", data.get("code", "No route returned")))
                route = data["routes"][0]
                route_popup = (popup + f"<br>Route distance: {route['distance'] / 1000:.2f} km"
                               f"<br>Route time: {route['duration'] / 60:.1f} min")
                folium.PolyLine(
                    [(lat, lon) for lon, lat in route["geometry"]["coordinates"]],
                    color="blue", weight=3, opacity=0.65,
                    tooltip=f"HDB {origin_label} → childcare {destination_label}",
                    popup=route_popup,
                ).add_to(route_layer)
                routes_drawn += 1
            except (requests.RequestException, ValueError, KeyError, IndexError) as exc:
                route_errors.append(f"HDB {gdf_origenes.index[i]}: {exc}")
                popup += "<br>Route geometry unavailable"
        else:
            no_finite_time += 1
            popup += "<br>No finite travel times in matrix (unreachable or failed query)"

        folium.Marker(
            [origin[1], origin[0]], popup=popup,
            icon=folium.Icon(color="blue", icon="home")
        ).add_to(origin_layer)

folium.LayerControl().add_to(m)
m.save("routes_map.html")
print(f"Saved routes_map.html: {routes_drawn:,} routes; "
      f"{no_finite_time:,} origins without finite times; {len(route_errors):,} route errors.")
for error in route_errors[:5]:
    print(error)

m


Displaying 100/10,741 HDB origins and 1,925 childcare destinations.


Loading walking routes: 100%|██████████| 100/100 [00:00<00:00, 188.18origin/s]


Saved routes_map.html: 100 routes; 0 origins without finite times; 0 route errors.
